# Traffic Signal RL Reproducibility Notebook

This notebook is the main reproducibility entry point for the Traffic Signal Control using Reinforcement Learning project.

It verifies the project structure, installs CityFlow from source with the required CMake compatibility patches, runs available experiment scripts, and summarizes generated result files.

**Before opening this notebook, create and activate the Conda environment:**

```bash
conda env create -f environment.yml
conda activate traffic-rl
python -m ipykernel install --user --name traffic-rl --display-name "traffic-rl"
```
Then select the `traffic-rl` kernel for this notebook. If you do not see it, restart your editor.


## 1. Locate the Project Root

This cell finds the repository root by searching for common project files and folders. It also adds the project root to `PYTHONPATH` so local modules can be imported.


In [1]:
from pathlib import Path
import os
import sys
import subprocess
import shutil
import textwrap
import json
import glob

NOTEBOOK_START = Path.cwd().resolve()

PROJECT_MARKERS = [
    "environment.yml",
    "env/cityflow_env.py",
    "scripts",
]

def find_project_root(start: Path) -> Path:
    for candidate in [start] + list(start.parents):
        if any((candidate / marker).exists() for marker in PROJECT_MARKERS):
            return candidate
    return start

PROJECT_ROOT = find_project_root(NOTEBOOK_START)
os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

existing_pythonpath = os.environ.get("PYTHONPATH", "")
os.environ["PYTHONPATH"] = str(PROJECT_ROOT) + (os.pathsep + existing_pythonpath if existing_pythonpath else "")

print("Notebook started in:", NOTEBOOK_START)
print("Project root:", PROJECT_ROOT)
print("Python executable:", sys.executable)


Notebook started in: /Users/achakrav6/Desktop/new-repo-asi
Project root: /Users/achakrav6/Desktop/new-repo-asi
Python executable: /opt/homebrew/Caskroom/miniconda/base/envs/traffic-rl/bin/python


## 2. Verify Expected Repository Structure

This confirms that the main project folders and files are present. Missing optional folders are reported as warnings instead of stopping the notebook.


In [2]:
expected_paths = [
    "environment.yml",
    "env/cityflow_env.py",
    "scripts",
    "scripts/baseline",
    "scripts/q_learning_experiments",
    "scripts/stable_baselines",
    "scripts/dqn_lstm",
    "scripts/lstm_vector",
]

missing = []
for rel_path in expected_paths:
    path = PROJECT_ROOT / rel_path
    status = "FOUND" if path.exists() else "MISSING"
    print(f"{status:8} {rel_path}")
    if not path.exists():
        missing.append(rel_path)

if missing:
    print("Some expected paths are missing. This is okay if those parts were not included in your final repo.")
else:
    print("Repository structure looks good.")


FOUND    environment.yml
FOUND    env/cityflow_env.py
FOUND    scripts
FOUND    scripts/baseline
FOUND    scripts/q_learning_experiments
FOUND    scripts/stable_baselines
FOUND    scripts/dqn_lstm
FOUND    scripts/lstm_vector
Repository structure looks good.


## 3. Helper Functions

These helpers run shell commands and Python scripts from the project root while printing clear output.


In [9]:
from types import SimpleNamespace


def run(cmd, cwd=PROJECT_ROOT, check=True, env=None):
    merged_env = os.environ.copy()
    merged_env.setdefault("PYTHONUNBUFFERED", "1")
    if env:
        merged_env.update(env)
    print("$", " ".join(map(str, cmd)))
    proc = subprocess.Popen(
        list(map(str, cmd)),
        cwd=cwd,
        env=merged_env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    chunks = []
    if proc.stdout is not None:
        for line in proc.stdout:
            print(line, end="")
            sys.stdout.flush()
            chunks.append(line)
    proc.wait()
    full_out = "".join(chunks)
    if check and proc.returncode != 0:
        raise RuntimeError(
            f"Command failed with exit code {proc.returncode}: {' '.join(map(str, cmd))}"
        )
    return SimpleNamespace(returncode=proc.returncode, stdout=full_out)


def run_python_script(script_path, required=False, extra_args=None):
    print(f"Running: {script_path}")
    script = PROJECT_ROOT / script_path
    if not script.exists():
        msg = f"Skipping missing script: {script_path}"
        if required:
            raise FileNotFoundError(msg)
        print(msg)
        return None
    args = [sys.executable, "-u", str(script)]
    if extra_args:
        args.extend(extra_args)
    return run(args, check=required)


## 4. Install or Verify CityFlow

CityFlow is built from source because this project uses the CityFlow simulator directly.

Newer CMake versions may fail on CityFlow's older CMake settings, so this cell patches all CityFlow CMake files that require versions older than 3.5 before installation.

This cell is idempotent: if `cityflow` is already importable, it skips the build.


In [10]:
import importlib.util
import re

FORCE_REINSTALL_CITYFLOW = False
CITYFLOW_DIR = PROJECT_ROOT / "CityFlow"
CITYFLOW_REPO = "https://github.com/cityflow-project/CityFlow.git"


def cityflow_is_available() -> bool:
    try:
        import cityflow  # noqa: F401
        return True
    except Exception:
        return False


def patch_cmake_versions(cityflow_dir: Path):
    cmake_files = list(cityflow_dir.rglob("CMakeLists.txt")) + list(cityflow_dir.rglob("*.cmake"))
    patched = []
    pattern = re.compile(r"cmake_minimum_required\(VERSION\s+([0-9]+(?:\.[0-9]+)*)\)")

    for file_path in cmake_files:
        text = file_path.read_text(errors="ignore")

        def replace(match):
            version = match.group(1)
            parts = tuple(int(p) for p in version.split("."))
            if parts < (3, 5):
                return "cmake_minimum_required(VERSION 3.5)"
            return match.group(0)

        new_text = pattern.sub(replace, text)
        if new_text != text:
            backup = file_path.with_suffix(file_path.suffix + ".bak")
            if not backup.exists():
                backup.write_text(text)
            file_path.write_text(new_text)
            patched.append(file_path.relative_to(cityflow_dir))

    print(f"Patched {len(patched)} CMake file(s).")
    for p in patched:
        print(" -", p)


if cityflow_is_available() and not FORCE_REINSTALL_CITYFLOW:
    import cityflow
    print("CityFlow is already installed.")
    print("cityflow module:", cityflow)
else:
    if not shutil.which("git"):
        raise RuntimeError("git is required. Install it through Conda or your system package manager.")
    if not shutil.which("cmake"):
        raise RuntimeError("cmake is required. It should be installed by environment.yml.")

    if not CITYFLOW_DIR.exists():
        run(["git", "clone", CITYFLOW_REPO, str(CITYFLOW_DIR)])
    else:
        print("CityFlow directory already exists:", CITYFLOW_DIR)

    run(["git", "submodule", "update", "--init", "--recursive"], cwd=CITYFLOW_DIR)
    patch_cmake_versions(CITYFLOW_DIR)

    for artifact in ["build", "dist"]:
        path = CITYFLOW_DIR / artifact
        if path.exists():
            shutil.rmtree(path)
    for egg_info in CITYFLOW_DIR.glob("*.egg-info"):
        shutil.rmtree(egg_info)

    run(
        [sys.executable, "-m", "pip", "install", "."],
        cwd=CITYFLOW_DIR,
        env={"CMAKE_ARGS": "-DCMAKE_POLICY_VERSION_MINIMUM=3.5"},
    )

    import cityflow
    print("CityFlow installed successfully.")
    print("cityflow module:", cityflow)


CityFlow is already installed.
cityflow module: <module 'cityflow' from '/opt/homebrew/Caskroom/miniconda/base/envs/traffic-rl/lib/python3.10/site-packages/cityflow.cpython-310-darwin.so'>


## 5. Create Output Folders

Experiment scripts write outputs to `results/`, `figures/`, `logs/`, and `models/`.


In [11]:
for folder in ["results", "figures", "logs", "models"]:
    (PROJECT_ROOT / folder).mkdir(exist_ok=True)
print("Output folders are ready.")


Output folders are ready.


## 6. Import the Custom CityFlow Environment

This verifies that the project environment wrapper can be imported.


In [12]:
try:
    from env.cityflow_env import CityFlowEnv
    print("Imported CityFlowEnv successfully:", CityFlowEnv)
except Exception as exc:
    print("Could not import CityFlowEnv.")
    print("Error:", repr(exc))
    raise


Imported CityFlowEnv successfully: <class 'env.cityflow_env.CityFlowEnv'>


## 7. Configure Experiment Run Mode

Use `QUICK_RUN = True` for a fast smoke test. Set `QUICK_RUN = False` to run the full available experiment pipeline.

Some scripts may take a long time depending on the number of episodes and the size of the CityFlow dataset.


In [18]:
QUICK_RUN = True
RUN_Q_LEARNING = True
RUN_STABLE_BASELINES = False if QUICK_RUN else True
RUN_DQN_LSTM = False if QUICK_RUN else True
RUN_LSTM_VECTOR = False if QUICK_RUN else True

print("QUICK_RUN:", QUICK_RUN)
print("RUN_Q_LEARNING:", RUN_Q_LEARNING)
print("RUN_STABLE_BASELINES:", RUN_STABLE_BASELINES)
print("RUN_DQN_LSTM:", RUN_DQN_LSTM)
print("RUN_LSTM_VECTOR:", RUN_LSTM_VECTOR)


QUICK_RUN: True
RUN_Q_LEARNING: True
RUN_STABLE_BASELINES: False
RUN_DQN_LSTM: False
RUN_LSTM_VECTOR: False


## 8. List Available Scripts

This makes the notebook robust to small repository differences by showing exactly which scripts are available.


In [13]:
script_files = sorted((PROJECT_ROOT / "scripts").rglob("*.py")) if (PROJECT_ROOT / "scripts").exists() else []
print(f"Found {len(script_files)} Python script(s) under scripts/.")
for script in script_files:
    print(script.relative_to(PROJECT_ROOT))


Found 31 Python script(s) under scripts/.
scripts/baseline/plot_baseline_results.py
scripts/baseline/run_baseline.py
scripts/dqn_lstm/collect_lstm_data.py
scripts/dqn_lstm/train.py
scripts/dqn_lstm/train_dqn_with_lstm.py
scripts/dqn_lstm/train_lstm.py
scripts/lstm_vector/lstm_vector_train.py
scripts/q_learning_experiments/bucketed_no_phase/train_bucketed_no_phase_default_reward.py
scripts/q_learning_experiments/bucketed_no_phase/train_bucketed_no_phase_delta_wait.py
scripts/q_learning_experiments/bucketed_no_phase/train_bucketed_no_phase_wait_small_vehicle.py
scripts/q_learning_experiments/bucketed_no_phase/train_bucketed_no_phase_wait_vehicle.py
scripts/q_learning_experiments/bucketed_with_phase/train_bucketed_with_phase_default_reward.py
scripts/q_learning_experiments/bucketed_with_phase/train_bucketed_with_phase_delta_wait.py
scripts/q_learning_experiments/bucketed_with_phase/train_bucketed_with_phase_wait_small_vehicle.py
scripts/q_learning_experiments/bucketed_with_phase/train_buc

## 9. Prepare Dataset Location for CityFlow Scripts

Some scripts expect the Hangzhou dataset either in the project root or inside the `CityFlow/` directory. This cell copies the dataset into `CityFlow/` if needed.


In [14]:
source_candidates = [
    PROJECT_ROOT / "hangzhou_datasets",
    PROJECT_ROOT / "data" / "hangzhou_datasets",
]

target_dir = PROJECT_ROOT / "CityFlow" / "hangzhou_datasets"

if target_dir.exists():
    print("Dataset already available at:", target_dir)
else:
    copied = False
    for source in source_candidates:
        if source.exists():
            shutil.copytree(source, target_dir)
            print(f"Copied dataset from {source} to {target_dir}")
            copied = True
            break
    if not copied:
        print("No Hangzhou dataset folder found in the expected locations.")
        print("If scripts fail with missing roadnet/flow/config files, check dataset paths in your repo.")


Dataset already available at: /Users/achakrav6/Desktop/new-repo-asi/CityFlow/hangzhou_datasets


## 10. Run Q-Learning Experiments

The Q-learning experiments are organized by state representation and reward function. Each block runs one group of experiments and saves its results to the `results/` folder.

Each Q-learning configuration (e.g., bucketed, capped, with/without phase, different reward functions) is run as a separate block.

- **Runtime:**  
  Each configuration typically takes **~20 minutes** to complete, depending on system performance.  
  Since multiple configurations are run sequentially, the full Q-learning section may take **1–2+ hours total**.

- **Where results are saved:**  
  All outputs are saved inside the `results/` directory, organized by: `results/<dataset_name>/<experiment_name>/`


---

### Output Files

Each experiment generates the following files:

#### 1. `q_learning_training.csv`
- Logs training progress across episodes
- Key columns include:
- `episode`
- `total_reward`
- `mean_reward_per_decision`
- `final_waiting`
- `avg_waiting`
- `final_avg_travel_time`
- `epsilon`

---

#### 2. `q_learning_trace.csv`
- Records a single evaluation run using the learned policy
- Each row corresponds to one decision step
- Includes:
- chosen action / phase
- state representation (e.g., bucketed queues)
- reward at each step
- total waiting time
- vehicle count
- average travel time

---

#### 3. `q_table.pkl`
- Serialized Q-table (learned policy)
- Can be loaded later for evaluation without retraining

---

#### 4. `runtime_config.json`
- The exact simulation configuration used for the run
- Ensures reproducibility

---

#### 5. `summary_all_datasets.csv` (generated after all runs, in a separate folder for the experiment)
- Aggregated summary across all datasets
- Includes:
- total reward
- final waiting time
- average waiting time
- travel time


### Bucketed No Phase

In [ ]:
bucketed_no_phase_scripts = [
    "scripts/q_learning_experiments/bucketed_no_phase/train_bucketed_no_phase_default_reward.py",
    "scripts/q_learning_experiments/bucketed_no_phase/train_bucketed_no_phase_delta_wait.py",
    "scripts/q_learning_experiments/bucketed_no_phase/train_bucketed_no_phase_wait_small_vehicle.py",
    "scripts/q_learning_experiments/bucketed_no_phase/train_bucketed_no_phase_wait_vehicle.py",
]

for script in bucketed_no_phase_scripts:
    run_python_script(script, required=True)

### Bucked With Phase

In [ ]:
bucketed_with_phase_scripts = [
    "scripts/q_learning_experiments/bucketed_with_phase/train_bucketed_with_phase_default_reward.py",
    "scripts/q_learning_experiments/bucketed_with_phase/train_bucketed_with_phase_delta_wait.py",
    "scripts/q_learning_experiments/bucketed_with_phase/train_bucketed_with_phase_wait_small_vehicle.py",
    "scripts/q_learning_experiments/bucketed_with_phase/train_bucketed_with_phase_wait_vehicle.py",
]

for script in bucketed_with_phase_scripts:
    run_python_script(script, required=True)

### Capped No Phase

In [ ]:
capped_no_phase_scripts = [
    "scripts/q_learning_experiments/capped_no_phase/train_capped_no_phase_default_reward.py",
    "scripts/q_learning_experiments/capped_no_phase/train_capped_no_phase_delta_wait.py",
    "scripts/q_learning_experiments/capped_no_phase/train_capped_no_phase_wait_small_vehicle.py",
    "scripts/q_learning_experiments/capped_no_phase/train_capped_no_phase_wait_vehicle.py",
]

for script in capped_no_phase_scripts:
    run_python_script(script, required=True)

### Capped With Phase

In [ ]:
capped_with_phase_scripts = [
    "scripts/q_learning_experiments/capped_with_phase/train_capped_with_phase_default_reward.py",
    "scripts/q_learning_experiments/capped_with_phase/train_capped_with_phase_delta_wait.py",
    "scripts/q_learning_experiments/capped_with_phase/train_capped_with_phase_wait_small_vehicle.py",
    "scripts/q_learning_experiments/capped_with_phase/train_capped_with_phase_wait_vehicle.py",
]

for script in capped_with_phase_scripts:
    run_python_script(script, required=True)

### Final Q-learning Comparison
This script compares the Hangzhou Q-learning model variants after the individual state and reward experiments have been run.

In [19]:
comparison_script = "scripts/q_learning_experiments/compare_hangzhou_models.py"

run_python_script(comparison_script, required=True)

Running: scripts/q_learning_experiments/compare_hangzhou_models.py
$ /opt/homebrew/Caskroom/miniconda/base/envs/traffic-rl/bin/python -u /Users/achakrav6/Desktop/new-repo-asi/scripts/q_learning_experiments/compare_hangzhou_models.py

Dataset: hangzhou_1x1_bc-tyc_18041607_1h
Config path: CityFlow/hangzhou_datasets/hangzhou_1x1_bc-tyc_18041607_1h/config.json
Intersection id: intersection_1_1
Lane ids: ['road_0_1_0_0', 'road_0_1_0_1', 'road_1_0_1_0', 'road_1_0_1_1', 'road_1_2_3_0', 'road_1_2_3_1', 'road_2_1_2_0', 'road_2_1_2_1']
Phase ids: [1, 2, 3, 4, 5, 6, 7, 8]

Comparison:
fixed_cycle                              | reward=  -3238.00 | avg_wait=   22.49 | final_wait=   38.00 | travel_time=   76.64
bucketed_no_phase_default                | reward=  -1817.00 | avg_wait=   12.62 | final_wait=   58.00 | travel_time=   58.30
bucketed_no_phase_delta_wait             | reward=  -1072.00 | avg_wait=    7.44 | final_wait=   21.00 | travel_time=   57.14
bucketed_no_phase_wait_small_vehicle     

namespace(returncode=0,
          stdout="\n======================================================================\nDataset: hangzhou_1x1_bc-tyc_18041607_1h\nConfig path: CityFlow/hangzhou_datasets/hangzhou_1x1_bc-tyc_18041607_1h/config.json\nIntersection id: intersection_1_1\nLane ids: ['road_0_1_0_0', 'road_0_1_0_1', 'road_1_0_1_0', 'road_1_0_1_1', 'road_1_2_3_0', 'road_1_2_3_1', 'road_2_1_2_0', 'road_2_1_2_1']\nPhase ids: [1, 2, 3, 4, 5, 6, 7, 8]\n\nComparison:\nfixed_cycle                              | reward=  -3238.00 | avg_wait=   22.49 | final_wait=   38.00 | travel_time=   76.64\nbucketed_no_phase_default                | reward=  -1817.00 | avg_wait=   12.62 | final_wait=   58.00 | travel_time=   58.30\nbucketed_no_phase_delta_wait             | reward=  -1072.00 | avg_wait=    7.44 | final_wait=   21.00 | travel_time=   57.14\nbucketed_no_phase_wait_small_vehicle     | reward=   -906.00 | avg_wait=    6.29 | final_wait=   21.00 | travel_time=   56.28\nbucketed_no_phase_wait

## 11. Run Stable-Baselines3 Experiments

This section runs DQN, PPO, and A2C scripts if they are present and enabled.


In [20]:
if RUN_STABLE_BASELINES:
    sb3_candidates = [
        "scripts/stable_baselines/train_dqn.py",
        "scripts/stable_baselines/train_ppo.py",
        "scripts/stable_baselines/train_a2c.py",
        "scripts/stable_baselines/compare_sb3_models.py",
    ]
    for candidate in sb3_candidates:
        run_python_script(candidate, required=False)
else:
    print("Skipping Stable-Baselines3 experiments. Set RUN_STABLE_BASELINES=True to run them.")


Skipping Stable-Baselines3 experiments. Set RUN_STABLE_BASELINES=True to run them.


## 12. Run DQN + LSTM Pipeline

This section runs temporal-state experiments if the scripts are available and enabled.


In [21]:
if RUN_DQN_LSTM:
    dqn_lstm_steps = [
        "scripts/dqn_lstm/collect_lstm_data.py",
        "scripts/dqn_lstm/train_lstm.py",
        "scripts/dqn_lstm/train_dqn_with_lstm.py",
    ]
    for step in dqn_lstm_steps:
        run_python_script(step, required=False)
else:
    print("Skipping DQN + LSTM pipeline. Set RUN_DQN_LSTM=True to run it.")


Skipping DQN + LSTM pipeline. Set RUN_DQN_LSTM=True to run it.


## 13. Run LSTM Vector Experiments

This section runs LSTM vector representation experiments if available and enabled.


In [22]:
if RUN_LSTM_VECTOR:
    lstm_vector_candidates = [
        "scripts/lstm_vector/lstm_vector_train.py",
        "scripts/lstm_vector/train_lstm_vector.py",
        "scripts/lstm_vector/evaluate_lstm_vector.py",
    ]
    for candidate in lstm_vector_candidates:
        run_python_script(candidate, required=False)
else:
    print("Skipping LSTM vector experiments. Set RUN_LSTM_VECTOR=True to run them.")


Skipping LSTM vector experiments. Set RUN_LSTM_VECTOR=True to run them.


## 14. Load and Summarize Result CSV Files

This cell reads all CSV files generated in `results/`, displays their final rows, and creates a combined summary table.


In [23]:
import pandas as pd
from IPython.display import display

csv_files = sorted((PROJECT_ROOT / "results").rglob("*.csv"))
print(f"Found {len(csv_files)} CSV file(s).")

summary_rows = []

for csv_file in csv_files:
    try:
        df = pd.read_csv(csv_file)
        print("===", csv_file.relative_to(PROJECT_ROOT), "===")
        display(df.tail(3))

        row = {"file": str(csv_file.relative_to(PROJECT_ROOT)), "rows": len(df)}
        if len(df) > 0:
            last = df.iloc[-1]
            for col in df.columns:
                value = last[col]
                if pd.api.types.is_numeric_dtype(df[col]):
                    row[f"final_{col}"] = value
        summary_rows.append(row)
    except Exception as exc:
        print(f"Could not read {csv_file}: {exc}")

summary_df = pd.DataFrame(summary_rows)
if not summary_df.empty:
    summary_path = PROJECT_ROOT / "results" / "final_comparison_summary.csv"
    summary_df.to_csv(summary_path, index=False)
    print("Saved summary to:", summary_path.relative_to(PROJECT_ROOT))
    display(summary_df)
else:
    print("No result CSVs found yet. Run experiment cells above or check script output paths.")


Found 135 CSV file(s).
=== results/baseline/fixed_ep1.csv ===


,step,sim_time,reward,total_waiting,vehicle_count,avg_travel_time
37,37,190.0,-56.4,38.0,204,57.385965
38,38,195.0,-52.8,34.0,208,58.164530
39,39,200.0,-52.2,36.0,212,58.922917


=== results/baseline/fixed_ep2.csv ===


,step,sim_time,reward,total_waiting,vehicle_count,avg_travel_time
37,37,190.0,-56.4,38.0,204,57.385965
38,38,195.0,-52.8,34.0,208,58.164530
39,39,200.0,-52.2,36.0,212,58.922917


=== results/baseline/fixed_ep3.csv ===


...

,model,mean_total_reward,mean_final_waiting,mean_avg_waiting,mean_avg_travel_time
0,dqn,-2965.6,153.0,71.075,60.233333
1,ppo,-3066.1,155.0,73.400,61.125000
2,a2c,-3675.9,208.0,90.175,65.066667


=== results/stable_baselines/sb3_comparison_all_datasets_states_rewards.csv ===


,dataset,model,state_mode,reward_mode,mean_total_reward,mean_final_waiting,mean_avg_waiting,mean_avg_travel_time
141,hangzhou_1x1_bc-tyc_18041610_1h,dqn,bucketed_with_phase,delta_wait,-20.0,20.0,12.701389,87.990172
142,hangzhou_1x1_bc-tyc_18041610_1h,ppo,bucketed_with_phase,delta_wait,-23.0,23.0,11.125000,81.265356
143,hangzhou_1x1_bc-tyc_18041610_1h,a2c,bucketed_with_phase,delta_wait,-25.0,25.0,13.041667,85.461916


Saved summary to: results/final_comparison_summary.csv


,file,rows,final_step,final_sim_time,final_reward,final_total_waiting,final_vehicle_count,final_avg_travel_time,final_eval_wait_only_reward,final_avg_waiting,...,final_num_decisions,final_run,final_mean_total_reward,final_mean_final_waiting,final_mean_avg_waiting,final_mean_avg_travel_time,final_baseline_mean_total_reward,final_baseline_mean_final_waiting,final_baseline_mean_avg_waiting,final_baseline_mean_avg_travel_time
0,results/baseline/fixed_ep1.csv,40,39.0,200.0,-52.2,36.0,212.0,58.922917,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,results/baseline/fixed_ep2.csv,40,39.0,200.0,-52.2,36.0,212.0,58.922917,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,results/baseline/fixed_ep3.csv,40,39.0,200.0,-52.2,36.0,212.0,58.922917,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,results/baseline/fixed_ep4.csv,40,39.0,200.0,-52.2,36.0,212.0,58.922917,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,results/baseline/fixed_ep5.csv,40,39.0,200.0,-52.2,36.0,212.0,58.922917,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
130,results/reward_experiments/training_wait_only.csv,2000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,73.200,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
131,results/reward_experiments/training_wait_small...,2000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,72.825,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
132,results/reward_experiments/training_wait_vehic...,2000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,73.100,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
133,results/stable_baselines/sb3_comparison.csv,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,-3675.9,208.0,90.175000,65.066667,NaN,NaN,NaN,NaN


## 15. Generate Final Comparison Plot

This cell creates a simple final comparison chart from the combined summary when numeric result columns are available.


In [ ]:
import matplotlib.pyplot as plt

if 'summary_df' in globals() and not summary_df.empty:
    numeric_cols = [col for col in summary_df.columns if col.startswith("final_") and pd.api.types.is_numeric_dtype(summary_df[col])]
    print("Numeric final metric columns:", numeric_cols)

    if numeric_cols:
        metric = numeric_cols[0]
        plot_df = summary_df[["file", metric]].dropna().copy()
        if not plot_df.empty:
            plot_df = plot_df.sort_values(metric)
            plt.figure(figsize=(12, max(4, 0.35 * len(plot_df))))
            plt.barh(plot_df["file"], plot_df[metric])
            plt.xlabel(metric)
            plt.ylabel("Experiment result file")
            plt.title(f"Final comparison by {metric}")
            plt.tight_layout()
            output_path = PROJECT_ROOT / "figures" / "final_comparison_plot.png"
            plt.savefig(output_path, dpi=200)
            plt.show()
            print("Saved plot to:", output_path.relative_to(PROJECT_ROOT))
        else:
            print("No non-null values available for plotting.")
    else:
        print("No numeric final metrics found for plotting.")
else:
    print("summary_df is empty or unavailable.")


## 17. Reproducibility Checklist

By the end of this notebook, verify that:

1. The Conda environment was created from `environment.yml`.
2. CityFlow installed and imported successfully.
3. `CityFlowEnv` imported successfully.
4. Experiment scripts were discovered under `scripts/`.
5. Result CSV files were generated or loaded from `results/`.
6. A final comparison summary was saved to `results/final_comparison_summary.csv`.

For a quick smoke test, keep `QUICK_RUN = True`. For the full reproducibility run, set `QUICK_RUN = False` and enable the experiment families you want to run.
